# 02 - Xây dựng đặc trưng

Notebook này tạo ma trận đặc trưng số từ hồ sơ tài khoản Cresci-2017.

## Bước 0 - Cấu hình đường dẫn local hoặc Google Colab

Nếu chạy trên Colab, hãy đặt project tại `/content/drive/MyDrive/bot-detection-project`.

In [1]:
from pathlib import Path
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/bot-detection-project')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: D:\Data mining\AnomalyDetection-main


In [2]:
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"

## Bước 1 - Tải dữ liệu và xây dựng đặc trưng

In [3]:
from src.preprocess import load_cresci2017
from src.features import build_features

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = load_cresci2017(RAW_DATA_DIR)
X, y = build_features(df)

[dữ liệu] Đang tìm Cresci-2017 tại: D:\Data mining\AnomalyDetection-main\data\raw
[dữ liệu] Đã tải genuine_accounts: 3,474 dòng, 44 cột
[dữ liệu] Đã tải social_spambots_1: 991 dòng, 43 cột
[dữ liệu] Đã tải social_spambots_2: 3,457 dòng, 42 cột
[dữ liệu] Đã tải social_spambots_3: 464 dòng, 43 cột
[dữ liệu] Đã tải traditional_spambots_1: 1,000 dòng, 42 cột
[dữ liệu] Đã tải traditional_spambots_4: 1,128 dòng, 42 cột
[dữ liệu] Kích thước sau khi gộp: 10,514 dòng x 44 cột
[dữ liệu] Phân phối nhãn:
label
bot           7040
người thật    3474
Name: count, dtype: int64
[đặc trưng] Bắt đầu xây dựng đặc trưng...
[đặc trưng] Đã tạo followers_friends_ratio


[đặc trưng] Đã tạo account_age_days
[đặc trưng] Đã tạo tweets_per_day
[features] Created statuses_followers_ratio and statuses_friends_ratio
[features] Created favourites_statuses_ratio
[đặc trưng] Đã tạo has_profile_image
[đặc trưng] Đã tạo has_description
[đặc trưng] Đã tạo name_length
[đặc trưng] Đã tạo screen_name_digit_ratio và screen_name_has_digits
[features] Created spam_keyword_count
[đặc trưng] Đã loại 25 cột định danh, văn bản hoặc metadata
[đặc trưng] Ma trận cuối: 10,514 dòng x 31 cột
[đặc trưng] Số giá trị thiếu cần xử lý sau khi chia train/test: 67,363


C:\Users\Danh\AppData\Local\Temp\ipykernel_14436\2846507920.py:9: UserWarning: Loại cột hoàn toàn rỗng: follow_request_sent, notifications, contributors_enabled, following, has_profile_image
  X, y = build_features(df)


## Bước 2 - Kiểm tra ma trận đặc trưng

In [4]:
print(f'Kích thước X: {X.shape}')
print(f'Kích thước y: {y.shape}')
print('\nDanh sách đặc trưng:')
for column in X.columns:
    print(f'- {column}')
print(f'\nTổng số giá trị thiếu: {int(X.isna().sum().sum()):,}')
display(X.head())

Kích thước X: (10514, 31)
Kích thước y: (10514,)

Danh sách đặc trưng:
- statuses_count
- followers_count
- friends_count
- favourites_count
- listed_count
- default_profile
- default_profile_image
- geo_enabled
- profile_use_background_image
- utc_offset
- is_translator
- protected
- verified
- followers_friends_ratio
- friends_followers_ratio
- friends_followers_gap
- account_age_days
- tweets_per_day
- statuses_followers_ratio
- statuses_friends_ratio
- favourites_statuses_ratio
- has_description
- description_length
- description_has_spam_keyword
- description_has_url
- name_length
- screen_name_length
- screen_name_digit_ratio
- screen_name_has_digits
- screen_name_has_spam_keyword
- spam_keyword_count

Tổng số giá trị thiếu: 67,363


,statuses_count,followers_count,friends_count,favourites_count,listed_count,default_profile,default_profile_image,geo_enabled,profile_use_background_image,utc_offset,...,has_description,description_length,description_has_spam_keyword,description_has_url,name_length,screen_name_length,screen_name_digit_ratio,screen_name_has_digits,screen_name_has_spam_keyword,spam_keyword_count
0,2177,208,332,265,1,NaN,NaN,1.0,NaN,NaN,...,1,21,0,0,15,8,0.500000,1,0,0
1,2660,330,485,3972,5,1.0,NaN,1.0,1.0,32400.0,...,1,48,0,0,5,8,0.500000,1,0,0
2,1254,166,177,1185,0,NaN,NaN,NaN,1.0,-14400.0,...,1,34,0,0,8,9,0.222222,1,0,0
3,202968,2248,981,60304,101,NaN,NaN,1.0,1.0,-7200.0,...,1,58,0,0,17,11,0.000000,0,0,0
4,82,21,79,5,0,1.0,NaN,NaN,1.0,NaN,...,1,13,0,0,8,15,0.533333,1,0,0


## Bước 3 - Lưu dữ liệu đã xử lý

Giá trị thiếu được giữ lại ở bước này. Notebook modeling sẽ fit bộ điền trung vị trên tập train để tránh leakage.

In [5]:
features_df = X.copy()
features_df['label'] = y.to_numpy()
output_file = PROCESSED_DIR / 'features.csv'
features_df.to_csv(output_file, index=False)
print(f'Đã lưu dữ liệu xử lý tại: {output_file}')

Đã lưu dữ liệu xử lý tại: D:\Data mining\AnomalyDetection-main\data\processed\features.csv
